# Preprocesamiento

In [1]:
from pathlib import Path

import pandas as pd

ruta = Path("data") / "dataset.csv"
datos = pd.read_csv(ruta, na_values=["NA"])

componentes = ["year", "month", "day", "hour"]
datos["fecha_hora"] = (
    pd.to_datetime(datos[componentes], errors="raise")
    .dt.tz_localize("Asia/Shanghai")
)

datos = datos.sort_values("fecha_hora").reset_index(drop=True)

display(datos[["fecha_hora", "pm2.5", "TEMP", "cbwd"]].head())

,fecha_hora,pm2.5,TEMP,cbwd
0,2010-01-01 00:00:00+08:00,NaN,-11.0,NW
1,2010-01-01 01:00:00+08:00,NaN,-12.0,NW
2,2010-01-01 02:00:00+08:00,NaN,-11.0,NW
3,2010-01-01 03:00:00+08:00,NaN,-14.0,NW
4,2010-01-01 04:00:00+08:00,NaN,-12.0,NW


Comprobamos la coherencia entre temperatura y el punto de rocío. DEWP debe ser menor o igual que la temperatura de aire TEMP. 

In [2]:
rocio_superior = datos["DEWP"] > datos["TEMP"]

discrepancias = datos.loc[
    rocio_superior, ["fecha_hora", "TEMP", "DEWP"]
].copy()
discrepancias["diferencia_C"] = discrepancias["DEWP"] - discrepancias["TEMP"]

print(f"Registros con DEWP superior a TEMP: {len(discrepancias)}")
display(discrepancias)

Registros con DEWP superior a TEMP: 1


,fecha_hora,TEMP,DEWP,diferencia_C
6727,2010-10-08 07:00:00+08:00,12.0,13,1.0


La comprobación detectó un único registro en el que el punto de rocío supera en 1 °C a la temperatura del aire.

Siguiendo la decisión de la primera entrega, mantenemos TEMP y ajustamos DEWP a 12 °C, el cambio mínimo que satisface la regla manteniendo fija la temperatura.

`DEWP_original` y `DEWP_corregido` documentan el ajuste en `datos_limpios`. El análisis usa `DEWP` con el valor ajustado.

In [3]:
datos_limpios = datos.copy()

fecha_ajuste = pd.Timestamp("2010-10-08 07:00", tz="Asia/Shanghai")
fila_ajuste = (
    datos_limpios["fecha_hora"].eq(fecha_ajuste)
    & datos_limpios["DEWP"].gt(datos_limpios["TEMP"])
)

datos_limpios["DEWP_original"] = datos_limpios["DEWP"]
datos_limpios["DEWP_corregido"] = fila_ajuste
datos_limpios.loc[fila_ajuste, "DEWP"] = datos_limpios.loc[fila_ajuste, "TEMP"]

display(datos_limpios.loc[fila_ajuste, ["fecha_hora", "DEWP_original", "DEWP", "TEMP"]])

,fecha_hora,DEWP_original,DEWP,TEMP
6727,2010-10-08 07:00:00+08:00,13,12,12.0


In [4]:
datos_analisis = (
    datos_limpios.iloc[24:]
    .drop(columns=["No", "DEWP_original", "DEWP_corregido"])
    .reset_index(drop=True)
)

datos_analisis.head()

,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir,fecha_hora
0,2010,1,2,0,129.0,-16,-4.0,1020.0,SE,1.79,0,0,2010-01-02 00:00:00+08:00
1,2010,1,2,1,148.0,-15,-4.0,1020.0,SE,2.68,0,0,2010-01-02 01:00:00+08:00
2,2010,1,2,2,159.0,-11,-5.0,1021.0,SE,3.57,0,0,2010-01-02 02:00:00+08:00
3,2010,1,2,3,181.0,-7,-5.0,1022.0,SE,5.36,1,0,2010-01-02 03:00:00+08:00
4,2010,1,2,4,138.0,-7,-5.0,1022.0,SE,6.25,2,0,2010-01-02 04:00:00+08:00
